In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.signal import savgol_filter, butter, filtfilt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm, TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.path import Path
from matplotlib.patches import PathPatch
%load_ext rpy2.ipython

Read in data and set plotting params

In [ ]:
sns.set_theme(style="ticks", context="talk", font="Arial")

plt.rcParams.update({
    # Font + text sizes
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],  # fallbacks"

    # Line widths
    "axes.linewidth": 0.5,   
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,     
    "ytick.major.size": 2,

    # --- Keep text editable in exports ---
    "svg.fonttype": "none",   # SVG: keep text as <text> elements, not paths
    "pdf.fonttype": 42,       # PDF: embed fonts as TrueType, editable
    "ps.fonttype": 42,        # EPS: same as above
    "text.usetex": False,     # don’t force LaTeX (avoids path-conversion)
})

In [ ]:
df = pd.read_csv(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\ResidualTrialDF_Reward.csv')
df['Session'] = np.where(df['Session'] == 'Reward3', 'Reward', 'Monster')
df["Session"] = pd.Categorical(df["Session"], categories=['Reward', 'Monster'], ordered=True)

successdf = df.groupby(['Animal', 'Session', 'Trial'], observed=True).first().reset_index()[['Animal', 'Session', 'Trial', 'Time to Reward', 'Time to Monster']]
successdf['Rewarded'] = np.where(successdf['Time to Reward'].notna(), 1, 0)
successdf = successdf.groupby(['Animal', 'Session'], observed=True).agg("mean").reset_index()
goodanimals  = successdf[(successdf['Session'] == 'Reward') & (successdf['Rewarded'] >= 0.8)]['Animal'].to_list()

df = df[df['Animal'].isin(goodanimals)]
df['Site'] = df['Region']
df['Region'] = df['Site'].str[:2]

df = (df.sort_values(["Trial","Animal","Site","Time to Shelter"],
                     kind="mergesort", na_position="last")
        .reset_index(drop=True))

df['Outcome'] = np.where(df['Time to Reward'].notna(), 'Rewarded', 'Unrewarded')
df['MonsterCharge'] = np.where(df['Time to Monster'].notna(), 'Charge', 'NoCharge')
df['Outcome_Monster'] = df['Outcome']+df['MonsterCharge']

df['v_reward_radial'] = df['v_reward_radial']*-1

df = df[~((df['Region']=='TS')&(df['Animal']=='Wanda'))].copy()
df = df[~((df['Site']=='TSR')&(df['Animal']=='Chester'))].copy()

In [ ]:
def lowpass(x, fs, cutoff_hz=0.3, order=3):
    b, a = butter(order, cutoff_hz / (fs / 2.0), btype="low")
    return filtfilt(b, a, np.asarray(x, float))


In [ ]:
parts = []
for g_names, g, in df.groupby(['Outcome_Monster', 'Session', 'Animal', 'Trial', 'Site']):
    outcome, session = g_names[0], g_names[1]
    fs  = float(1.0 / g['dt'].iloc[0]) 
    win = int(round(fs * 1.5)); win += 1 - (win % 2) 
    win = max(win, 3)

    if len(g) < win:
        continue

    g['filtered_x']   = savgol_filter(g['X'].to_numpy(), win, 1, mode='interp')
    g['filtered_y']   = savgol_filter(g['Y'].to_numpy(), win, 1, mode='interp')
    dy                = savgol_filter(g['X'].to_numpy(), win, 1, deriv=1, delta=1/fs, mode='interp')
    g['d_filtered_x'] = lowpass(dy, fs, cutoff_hz=0.3)   
    g.loc[abs(g['d_filtered_x']) <2, 'd_filtered_x' ] = 0

    g['approach'] = np.nan
    g['approach'] = g['approach'].astype('object')
    g.loc[g['filtered_x'] < 30,  'approach'] = 'Distal Approach' #0 #Distal Approach
    g.loc[g['filtered_x'] >= 30, 'approach'] = 'Proximal Approach' #20 #Proximal Approach
    g.loc[g['filtered_x'] >= 43, 'approach'] = 'Past Spout' #60 #Go past Spout
    g.loc[g['d_filtered_x'] < 0, 'approach'] = 'Retreat' #40 # retreat overrides bands

    rewardix = g['Time to Reward'].abs().idxmin()
    monsterix = (g['Time to Monster'].abs()).idxmin()
    consumptionix = np.nanmin([g.loc[(g.index > rewardix) & (g['approach'] == 'Retreat')].index.min(), g.loc[(g.index > rewardix) & (g['approach'] == 'Past Spout')].index.min()])
    g.loc[rewardix:consumptionix, 'approach'] = 'Licking'

    g.loc[(g['filtered_x'] <= 0), 'approach'] = np.nan

    lab  = g['approach'].to_numpy() 
    prev = np.concatenate([lab[:1], lab[:-1]])

    transitions = (prev == 'Retreat') & (lab != 'Retreat')
    g['approach_counter'] = np.cumsum(transitions.astype(int))

    parts.append(g)
    
df = pd.concat(parts, axis=0).reset_index(drop=True)

df = df[df['approach'].notna()]

parts = []
for gname, g in df.groupby(['approach_counter', 'Outcome_Monster', 'Session', 'Animal', 'Trial', 'Site']):
    rmin, rmax = np.nanmin(g['Time to Reward']), np.nanmax(g['Time to Reward'])
    mmin, mmax = np.nanmin(g['Time to Monster']), np.nanmax(g['Time to Monster'])
    if np.isnan(rmin):
        g['Reward'] = 'Unrewarded'
    elif ((rmin<0) & (rmax<0)):
        g['Reward'] = 'Unrewarded'
    elif ((rmin<0) & (rmax>0)):
        g['Reward'] = 'Rewarded'
    elif ((rmin>0) & (rmax>0)):
        g['Reward'] = 'PostRewarded'

    if np.isnan(mmin):
        g['Monster'] = 'NoCharge'
    elif ((mmin<0) & (mmax<0)):
        g['Monster'] = 'NoCharge'
    elif ((mmin<0) & (mmax>0)):
        g['Monster'] = 'MonsterCharge'
    elif ((mmin>0) & (mmax>0)):
        g['Monster'] = 'MonsterCharging'

    parts.append(g)
df = pd.concat(parts, axis=0).reset_index(drop=True)

Figure 5A

In [ ]:
def plot_trajectories(
    df,
    sessions=("Reward","Monster"), regions=("VS","TS"),
    region_col="Region",
    arena_xlim=(0, 78), arena_ylim=(-10, 10), invert_y=False,
    color_col="velocity", cmap_name=None, vmin=None, vmax=None,
    line_w=1.0, line_alpha=0.9,
    gate_col='Z', gate_z_thresh=2.0, gate_enable=True,
    remove_jumps=True, max_step=None, jump_k=3.0,
    panel_aspect=None, axs=None,
    rasterize_lines=False,
    add_colorbar=True,
 
    compact_svg=True, 
    color_bins=None,
    rdp_tol=0.10,
    keep_corners_deg=20.0,
    coord_decimals=1
):


    def _as_list(x):
        return x if isinstance(x, (list, tuple)) else [x]

    def _zscore(a):
        m = np.nanmean(a); s = np.nanstd(a)
        return (a - m) / (s if s and np.isfinite(s) else 1.0)

    def _robust_threshold(dist, k):
        med = np.nanmedian(dist)
        mad = np.nanmedian(np.abs(dist - med))
        return med + k * (1.4826 * mad if np.isfinite(mad) and mad > 0 else 0)

    def _split_indices_on_jumps(x, y, max_step_local, k):
        if len(x) < 2: return []
        dx, dy = np.diff(x), np.diff(y)
        dist = np.hypot(dx, dy)
        thr = _robust_threshold(dist, k) if max_step_local is None else float(max_step_local)
        cut_points = np.where(dist > thr)[0]
        segments, start = [], 0
        for cp in cut_points:
            if cp + 1 - start >= 2:
                segments.append((start, cp + 1))
            start = cp + 1
        if len(x) - start >= 2:
            segments.append((start, len(x)))
        return segments

    def _true_runs(mask_bool):
        if mask_bool.size == 0: return
        m = mask_bool.astype(np.int8)
        dm = np.diff(np.pad(m, (1, 1)))
        starts = np.where(dm == 1)[0]
        ends   = np.where(dm == -1)[0]
        for s, e in zip(starts, ends):
            yield s, e

    def _turn_angles_deg(pts):
        if pts.shape[0] < 3:
            return np.empty(0)
        v0 = pts[1:-1] - pts[:-2]
        v1 = pts[2:]   - pts[1:-1]
        n0 = np.linalg.norm(v0, axis=1)
        n1 = np.linalg.norm(v1, axis=1)
        good = (n0 > 1e-12) & (n1 > 1e-12)
        cosang = np.ones_like(n0)
        cosang[good] = (v0[good] * v1[good]).sum(1) / (n0[good] * n1[good])
        cosang = np.clip(cosang, -1.0, 1.0)
        return np.degrees(np.arccos(cosang))

    def _rdp(points, eps):
        n = points.shape[0]
        if eps <= 0 or n < 3:
            return points
        stack = [(0, n - 1)]
        keep = np.zeros(n, dtype=bool)
        keep[[0, n - 1]] = True
        while stack:
            s, e = stack.pop()
            if e - s <= 1: 
                continue
            p0, p1 = points[s], points[e]
            v = p1 - p0
            vn = float(np.linalg.norm(v))
            if vn < 1e-12:
                seg = points[s:e+1]
                if seg.shape[0] <= 2: 
                    continue
                dists = np.linalg.norm(seg - p0, axis=1)
                idx = s + int(np.argmax(dists)); dmax = float(dists[idx - s])
            else:
                w = points[s+1:e] - p0
                if w.shape[0] == 0: 
                    continue
                cross_z = v[0] * w[:, 1] - v[1] * w[:, 0]
                perp = np.abs(cross_z) / vn
                off = int(np.argmax(perp))
                dmax = float(perp[off]); idx = s + 1 + off
            if dmax > eps:
                keep[idx] = True
                stack.append((s, idx)); stack.append((idx, e))
        return points[keep]

    def _simplify_polyline(xr, yr):
        pts = np.column_stack([xr, yr])
        if keep_corners_deg is not None and pts.shape[0] >= 3:
            ang = _turn_angles_deg(pts)
            sharp_idx = np.where(ang >= keep_corners_deg)[0] + 1
            boundaries = np.unique(np.concatenate(([0], sharp_idx, [pts.shape[0]-1]))).astype(int)
            out = []
            for i in range(len(boundaries)-1):
                s, e = boundaries[i], boundaries[i+1]
                if e - s < 1: 
                    continue
                seg = pts[s:e+1]
                if rdp_tol > 0 and seg.shape[0] >= 3:
                    seg = _rdp(seg, rdp_tol)
                out.append(seg)
            if not out:
                return xr, yr
            pts = np.vstack(out)
        else:
            if rdp_tol > 0 and pts.shape[0] >= 3:
                pts = _rdp(pts, rdp_tol)
        if coord_decimals is not None:
            pts = np.round(pts, int(coord_decimals))
        return pts[:,0], pts[:,1]

    base_cmap = (cmap_name if hasattr(cmap_name, '__call__')
                 else mpl.cm.get_cmap(cmap_name or 'viridis'))
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    sessions, regions = _as_list(sessions), _as_list(regions)
    for r, ses in enumerate(sessions):
        for c, reg in enumerate(regions):
            ax = axs[r, c]
            sub = df[(df['Session'] == ses) & (df[region_col] == reg)]

            if compact_svg and isinstance(color_bins, int):
                bins = int(color_bins)
                lo = norm.vmin if norm.vmin is not None else np.nanmin(df[color_col])
                hi = norm.vmax if norm.vmax is not None else np.nanmax(df[color_col])
                edges = np.linspace(lo, hi, bins+1)
                bin_vertices = [[] for _ in range(bins)]
                bin_codes    = [[] for _ in range(bins)]
            else:
                patches = []

            for _, trial in sub.groupby('Trial'):
                x = trial['X'].to_numpy(float); y = trial['Y'].to_numpy(float)
                if x.size < 2: 
                    continue
                v = trial[color_col].to_numpy(float)
                chunks = _split_indices_on_jumps(x, y, max_step, jump_k) if remove_jumps else [(0, x.size)]
                tg = trial[gate_col].to_numpy(float) if gate_enable else None

                for (i0, i1) in chunks:
                    if i1 - i0 < 2: 
                        continue
                    xi, yi = x[i0:i1], y[i0:i1]
                    vi = v[i0:i1-1]
                    mask = np.ones_like(vi, dtype=bool)
                    if gate_enable:
                        z = _zscore(tg[i0:i1])
                        mask = 0.5*(z[:-1] + z[1:]) > gate_z_thresh
                        if not np.any(mask):
                            continue

                    for s_run, e_run in _true_runs(mask):
                        xr = xi[s_run:e_run+1]
                        yr = yi[s_run:e_run+1]
                        if xr.size < 2: 
                            continue
                        xr, yr = _simplify_polyline(xr, yr)
                        if xr.size < 2: 
                            continue

                        if compact_svg and isinstance(color_bins, int):
                            v_med = float(np.nanmedian(vi[s_run:e_run]))
                            bi = int(np.digitize(v_med, edges) - 1)
                            bi = max(0, min(bins-1, bi))
                            verts = np.column_stack([xr, yr])
                            codes = np.full(verts.shape[0], Path.LINETO, np.uint8); codes[0] = Path.MOVETO
                            bin_vertices[bi].append(verts); bin_codes[bi].append(codes)
                        else:
                            color = base_cmap(norm(float(np.nanmedian(vi[s_run:e_run]))))
                            verts = np.column_stack([xr, yr])
                            codes = np.full(verts.shape[0], Path.LINETO, np.uint8); codes[0] = Path.MOVETO
                            path = Path(verts, codes)
                            patches.append(PathPatch(
                                path, facecolor='none', edgecolor=color,
                                linewidth=line_w, alpha=line_alpha,
                                capstyle='round', joinstyle='round'
                            ))

            # draw
            if compact_svg and isinstance(color_bins, int):
                for bi in range(int(color_bins)):
                    if not bin_vertices[bi]:
                        continue
                    verts = np.concatenate(bin_vertices[bi], axis=0)
                    codes = np.concatenate(bin_codes[bi], axis=0)
                    path = Path(verts, codes)
                    color = base_cmap((bi + 0.5)/int(color_bins))
                    ax.add_patch(PathPatch(
                        path, facecolor='none', edgecolor=color,
                        linewidth=line_w, alpha=line_alpha,
                        capstyle='round', joinstyle='round'
                    ))
            else:
                for p in patches:
                    ax.add_patch(p)
            ax.set_xlim(*arena_xlim); ax.set_ylim(*arena_ylim)
            if invert_y: ax.invert_yaxis()
            ax.set_xlabel("X (cm)"); ax.set_ylabel("Y (cm)")
            ax.set_box_aspect(panel_aspect if panel_aspect else
                              (arena_ylim[1]-arena_ylim[0])/(arena_xlim[1]-arena_xlim[0]))

    if add_colorbar:
        fig = axs[0, 0].figure
        if isinstance(color_bins, int):
            boundaries = np.linspace(
                norm.vmin if norm.vmin is not None else np.nanmin(df[color_col]),
                norm.vmax if norm.vmax is not None else np.nanmax(df[color_col]),
                int(color_bins)+1
            )
            sm = mpl.cm.ScalarMappable(norm=mpl.colors.BoundaryNorm(boundaries, int(color_bins)),
                                       cmap=base_cmap)
        else:
            sm = mpl.cm.ScalarMappable(norm=norm, cmap=base_cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=axs, orientation="vertical", fraction=0.025, pad=0.02)
        cbar.set_label(color_col)

    return axs


In [ ]:
colors = ["#A94CFF", "#2C2B2B", "#FF5335"]
CMAP = LinearSegmentedColormap.from_list("blue_black_yellow", colors, N=256)


sessions = ("Reward", "Monster")
regions  = ("VS", "TS")

fig = plt.figure(figsize=(8, 4)) 
gs  = fig.add_gridspec(nrows=len(sessions), ncols=12, hspace=0.3, wspace=0.2)

axs = np.array([
    [fig.add_subplot(gs[0, 0:6]),  fig.add_subplot(gs[0, 6:12])],  # row 0 Reward
    [fig.add_subplot(gs[1, 0:6]),  fig.add_subplot(gs[1, 6:12])]   # row 1 Monster
])

plot_trajectories(
    df[(df['approach'] == 'Proximal Approach') | (df['approach'] == 'Distal Approach')], sessions=sessions, regions=regions, region_col="Region",
    axs=axs, color_col="v_reward_radial", cmap_name=CMAP,
    gate_col='Z', gate_z_thresh=1, gate_enable=False,
    panel_aspect=0.25, vmin = -30, vmax = 30, max_step=8, rasterize_lines=True, line_w=.3
)

fig.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\RadialVelocity_Trajectory_Approach_Gated_thinlines.svg',dpi=800)

Figure 5B

In [ ]:
animal = 'EIVSTS4'
trial = 19
session ='Reward'

sub = df.loc[
    (df['Animal'] == animal) &
    (df['Trial'] == trial) &
    (df['Session'] == session) &
    (df['Time to Reward'] < 0) &
    (df['Site'] == 'VS')
].sort_values('Time to Reward')


x = sub['Time to Reward'].to_numpy()
y = savgol_filter(sub['v_reward_radial'].to_numpy(), 11, 1, mode='interp')
z = sub['Z'].to_numpy()

points = np.array([x, y]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)

lc = LineCollection(
    segments,
    cmap='viridis',
    norm=plt.Normalize(vmin=-1, vmax=2)   # set z range here
)
lc.set_array(z[:-1])   # one z value per segment
lc.set_linewidth(2)

fig, ax = plt.subplots()
ax.add_collection(lc)
ax.autoscale()
plt.colorbar(lc, ax=ax, label='z')

ax.set_xlabel('Time to Reward')
ax.set_ylabel('v_reward_radial')
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\EIVSTS4Trial19ReawardApproachVS.svg',dpi=800)
plt.show()

Figure 5C,5D,5E

In [ ]:
g = ["Region","Session", "Animal","Trial","Site"]
k = np.where(np.isclose(df.dt, 1/12, atol=1e-9), 12, 20)

num = [c for c in df.select_dtypes("number").columns if c not in g]
obj = [c for c in df.columns if c not in num+g]

df = (df.assign(_k=k, _i=df.groupby(g).cumcount(), bin=lambda d: d._i//d._k)
        .groupby(g+["bin"], as_index=False, observed=True)
        .agg({**{c:"mean" for c in num}, **{c:"first" for c in obj}})
        .drop(columns=["bin"])
        .assign(Z_lag1=lambda d: d.groupby(g)["Z"].shift(1))
     )

In [ ]:
%%R -i df -o velocity_approach_activity_trends
library(dplyr)
library(lme4)
library(emmeans)

df$Animal <- as.factor(df$Animal)
df$Trial <- as.factor(df$Trial) 
df$approach_counter <- as.factor(df$approach_counter)

velocity_approach_activity_model <- lmer(Z ~ v_reward_radial * Region  * approach * Session + Z_lag1 + (1 |Animal) + (1 | Trial) + (1 | Trial:approach_counter) + (1|Animal:Site), data = df)
acf(resid(velocity_approach_activity_model))


slopes <- emtrends(velocity_approach_activity_model,~  Region * approach | Session, var = "v_reward_radial")
velocity_approach_activity_trends <- as.data.frame(test(slopes, adjust = "bonferroni"))
print(velocity_approach_activity_trends)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

row_order        = ["Reward", "Monster"]
col_order        = ["VS", "TS"]
APPROACH_LEVELS  = ["Distal Approach", "Proximal Approach", "Retreat"]

X_LABEL          = "Velocity toward spout (v_reward_radial)"
Y_LABEL          = "Activity (Z)"

FIG_HEIGHT       = 3.6
FIG_ASPECT       = 1.05

AUTOZOOM_X       = True
AUTOZOOM_Y       = True

X_QLO, X_QHI     = 1, 99
Y_QLO, Y_QHI     = 1, 99

X_PAD_FRAC       = 0.08
Y_PAD_FRAC       = 0.08

MIN_XSPAN        = 10.0
MIN_YSPAN        = 0.6

X_CLAMP          = (-30, 30)
Y_CLAMP          = (-1.0, 1.0)

N_VEL_POINTS     = 200
RIBBON_ALPHA     = 0.18
BAND_MODE        = "auto"
SE_MULT          = 1.0

REGION_COLORS    = {"VS": "#1f77b4", "TS": "#E68600"}

# Density-colored scatter
SCATTER_SIZE     = 6
SCATTER_ALPHA    = 0.3
DENS_BINS        = 50
DENS_CMAP        = "viridis"
RASTERIZE_SCAT   = True

# Per-animal OLS lines
PLOT_ANIMAL_OLS      = True
ANIMAL_COL           = "Animal"
ANIMAL_MIN_PTS       = 25     # require enough samples per animal per panel
ANIMAL_LINE_COLOR    = "#000000" # neutral gray
ANIMAL_LINE_ALPHA    = 0.5
ANIMAL_LINE_LW       = 1.0

sns.set_theme(context="paper", style="ticks")
mpl.rcParams.update({
    "axes.titlesize": 9.5,
    "axes.labelsize": 9.5,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 150,
})


trend = velocity_approach_activity_trends.copy()
raw   = df.copy() 


for c in ["Region", "Session", "approach"]:
    if c in trend.columns:
        trend[c] = trend[c].astype(str)
    if c in raw.columns:
        raw[c] = raw[c].astype(str)

trend = trend[trend["Region"].isin(col_order)].copy()
raw   = raw[raw["Region"].isin(col_order)].copy()

trend["Session"]  = pd.Categorical(trend["Session"],  categories=row_order,       ordered=True)
trend["Region"]   = pd.Categorical(trend["Region"],   categories=col_order,       ordered=True)
trend["approach"] = pd.Categorical(trend["approach"], categories=APPROACH_LEVELS, ordered=True)

raw["Session"]  = pd.Categorical(raw["Session"],  categories=row_order,       ordered=True)
raw["Region"]   = pd.Categorical(raw["Region"],   categories=col_order,       ordered=True)
raw["approach"] = pd.Categorical(raw["approach"], categories=APPROACH_LEVELS, ordered=True)

trend["v_reward_radial.trend"] = pd.to_numeric(trend["v_reward_radial.trend"], errors="coerce")
for c in ["SE", "lower.CL", "upper.CL", "lower.CL.trend", "upper.CL.trend"]:
    if c in trend.columns:
        trend[c] = pd.to_numeric(trend[c], errors="coerce")
trend = trend.dropna(subset=["v_reward_radial.trend"])

raw["v_reward_radial"] = pd.to_numeric(raw["v_reward_radial"], errors="coerce")
raw["Z"]               = pd.to_numeric(raw["Z"], errors="coerce")
raw = raw[np.isfinite(raw["v_reward_radial"]) & np.isfinite(raw["Z"])].copy()

def make_panel(region, approach):
    return f"{region} — {approach}"

panel_order = [make_panel(r, a) for r in col_order for a in APPROACH_LEVELS]
trend["Panel"] = [make_panel(r, a) for r, a in zip(trend["Region"].astype(str), trend["approach"].astype(str))]
raw["Panel"]   = [make_panel(r, a) for r, a in zip(raw["Region"].astype(str),   raw["approach"].astype(str))]

trend["Panel"] = pd.Categorical(trend["Panel"], categories=panel_order, ordered=True)
raw["Panel"]   = pd.Categorical(raw["Panel"],   categories=panel_order, ordered=True)

def _get_ci_cols(df):
    if "lower.CL" in df.columns and "upper.CL" in df.columns:
        return "lower.CL", "upper.CL"
    if "lower.CL.trend" in df.columns and "upper.CL.trend" in df.columns:
        return "lower.CL.trend", "upper.CL.trend"
    return None, None

def _compute_limits(vals, qlo, qhi, pad_frac, min_span, clamp=None):
    vals = np.asarray(vals)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None

    lo = float(np.nanpercentile(vals, qlo))
    hi = float(np.nanpercentile(vals, qhi))
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return None

    span = hi - lo
    if span < min_span:
        med = float(np.nanmedian(vals))
        lo = med - 0.5 * min_span
        hi = med + 0.5 * min_span
        span = hi - lo

    lo -= pad_frac * span
    hi += pad_frac * span

    if clamp is not None:
        lo = max(clamp[0], lo)
        hi = min(clamp[1], hi)
        if hi - lo < 1e-9:
            return None

    return lo, hi

def _density01_hist2d_with_edges(x, y, xedges, yedges):
    x = np.asarray(x); y = np.asarray(y)
    if x.size < 5:
        return np.zeros_like(x, dtype=float)

    H, _, _ = np.histogram2d(x, y, bins=[xedges, yedges])
    xi = np.searchsorted(xedges, x, side="right") - 1
    yi = np.searchsorted(yedges, y, side="right") - 1
    xi = np.clip(xi, 0, H.shape[0] - 1)
    yi = np.clip(yi, 0, H.shape[1] - 1)

    d = H[xi, yi].astype(float)
    d = np.log1p(d)
    return (d - d.min()) / ((d.max() - d.min()) + 1e-12)

g = sns.FacetGrid(
    trend,
    row="Session",
    col="Panel",
    row_order=row_order,
    col_order=panel_order,
    height=FIG_HEIGHT,
    aspect=FIG_ASPECT,
    sharex=False,
    sharey=False
)

def _plot_panel(data, **kwargs):
    ax = plt.gca()

    sess   = str(data["Session"].iloc[0])
    region = str(data["Region"].iloc[0])
    appr   = str(data["approach"].iloc[0])
    color  = REGION_COLORS.get(region, "0.3")

    sub_raw = raw[
        (raw["Session"].astype(str) == sess) &
        (raw["Region"].astype(str)  == region) &
        (raw["approach"].astype(str)== appr)
    ].copy()

    # Per-panel limits from raw points
    xlim = _compute_limits(
        sub_raw["v_reward_radial"].to_numpy(),
        X_QLO, X_QHI, X_PAD_FRAC, MIN_XSPAN,
        clamp=X_CLAMP if AUTOZOOM_X else X_CLAMP
    ) if AUTOZOOM_X else X_CLAMP

    ylim = _compute_limits(
        sub_raw["Z"].to_numpy(),
        Y_QLO, Y_QHI, Y_PAD_FRAC, MIN_YSPAN,
        clamp=Y_CLAMP if AUTOZOOM_Y else Y_CLAMP
    ) if AUTOZOOM_Y else Y_CLAMP

    if xlim is None:
        xlim = X_CLAMP if X_CLAMP is not None else (-30, 30)
    if ylim is None:
        ylim = Y_CLAMP if Y_CLAMP is not None else (-1, 1)

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    xedges = np.linspace(xlim[0], xlim[1], DENS_BINS + 1)
    yedges = np.linspace(ylim[0], ylim[1], DENS_BINS + 1)

    if len(sub_raw) > 0:
        x = sub_raw["v_reward_radial"].to_numpy()
        y = sub_raw["Z"].to_numpy()

        msk = (x >= xlim[0]) & (x <= xlim[1]) & (y >= ylim[0]) & (y <= ylim[1])
        x2, y2 = x[msk], y[msk]

        if x2.size > 0:
            dens01 = _density01_hist2d_with_edges(x2, y2, xedges, yedges)
            order = np.argsort(dens01)
            ax.scatter(
                x2[order], y2[order],
                c=dens01[order],
                cmap=DENS_CMAP, vmin=0, vmax=1,
                s=SCATTER_SIZE,
                alpha=SCATTER_ALPHA,
                linewidths=0,
                rasterized=RASTERIZE_SCAT,
                zorder=1
            )

    # ---- per-animal OLS lines (raw) ----
    if PLOT_ANIMAL_OLS and (ANIMAL_COL in sub_raw.columns):
        sub_fit = sub_raw[
            (sub_raw["v_reward_radial"] >= xlim[0]) & (sub_raw["v_reward_radial"] <= xlim[1])
        ].copy()

        v_grid = np.linspace(xlim[0], xlim[1], N_VEL_POINTS)

        for animal, dfa in sub_fit.groupby(ANIMAL_COL):
            xa = dfa["v_reward_radial"].to_numpy()
            ya = dfa["Z"].to_numpy()
            msk = np.isfinite(xa) & np.isfinite(ya)
            xa, ya = xa[msk], ya[msk]
            if xa.size < ANIMAL_MIN_PTS:
                continue

            # OLS fit: y = b0 + b1*x
            b1, b0 = np.polyfit(xa, ya, 1)
            ax.plot(
                v_grid,
                b0 + b1 * v_grid,
                color=ANIMAL_LINE_COLOR,
                alpha=ANIMAL_LINE_ALPHA,
                lw=ANIMAL_LINE_LW,
                zorder=2
            )

    v_grid = np.linspace(xlim[0], xlim[1], N_VEL_POINTS)

    m = float(data["v_reward_radial.trend"].mean())
    lo_col, hi_col = _get_ci_cols(data)
    use_ci = (BAND_MODE == "CI") or (BAND_MODE == "auto" and lo_col is not None)

    if use_ci and lo_col is not None:
        m_lo = float(pd.to_numeric(data[lo_col], errors="coerce").mean())
        m_hi = float(pd.to_numeric(data[hi_col], errors="coerce").mean())
        if np.isfinite(m_lo) and np.isfinite(m_hi):
            ax.fill_between(
                v_grid,
                m_lo * v_grid,
                m_hi * v_grid,
                alpha=RIBBON_ALPHA,
                color=color,
                linewidth=0,
                zorder=3
            )
    else:
        se = float(pd.to_numeric(data["SE"], errors="coerce").mean()) if "SE" in data.columns else np.nan
        if np.isfinite(se):
            ax.fill_between(
                v_grid,
                (m - SE_MULT * se) * v_grid,
                (m + SE_MULT * se) * v_grid,
                alpha=RIBBON_ALPHA,
                color=color,
                linewidth=0,
                zorder=3
            )

    ax.plot(v_grid, m * v_grid, color=color, lw=2.0, zorder=4)

    ax.axhline(0, linestyle=(0, (4, 3)), linewidth=1.0, color="0.4", alpha=0.75, zorder=0)
    sns.despine(ax=ax, top=True, right=True, left=False, bottom=False)

    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(5))

g.map_dataframe(_plot_panel)

g.set_titles(col_template="Session = {row_name} | {col_name}")
for ax in g.axes.flatten():
    ax.set_xlabel("")
    ax.set_ylabel("")

g.figure.subplots_adjust(left=0.07, top=0.90, right=0.93, bottom=0.14, wspace=0.25, hspace=0.40)
fig = g.figure
fig.text(0.5, 0.06, X_LABEL, ha="center", va="center", fontsize=10)
fig.text(0.02, 0.52, Y_LABEL, rotation=90, ha="center", va="center", fontsize=10)

sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap=DENS_CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=g.axes, fraction=0.02, pad=0.01)
cbar.set_label("Relative point density", fontsize=9)
cbar.ax.tick_params(labelsize=8)
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Binned_Correlation_Velocity_IndividualFits.svg')
plt.show()

Figure 5F

In [ ]:
%%R -i df -o velocity_approach_activity_trends

library(dplyr)
library(lme4)
library(emmeans)
library(splines)

vars_needed <- c(
  "Z", "v_reward_radial", "speed",
  "Region", "approach", "Session",
  "Animal", "Trial", "approach_counter", "Site"
)

df2 <- df[complete.cases(df[, vars_needed]), ]

df2$Trial            <- factor(df2$Trial)
df2$approach_counter <- factor(df2$approach_counter)
df2$Animal           <- factor(df2$Animal)
df2$Site             <- factor(df2$Site)
df2$Region           <- factor(df2$Region)
df2$Session          <- factor(df2$Session)
df2$approach         <- factor(df2$approach)

#Regress out GLOBAL (nonlinear) speed effect from dopamine

k_df <- 4  # spline

m_speed_global <- lmer(
  Z ~ ns(log(speed), df = k_df) +
    (1 | Animal) + (1 | Trial) + (1 | Trial:approach_counter) + (1 | Animal:Site),
  data = df2, REML = FALSE,
  control = lmerControl(calc.derivs = FALSE)
)

df2$Z_resid <- resid(m_speed_global)


# Add AR(1) term on the residual series

df2 <- df2 %>%
  group_by(Animal, Site, Session, Trial, approach_counter, Region, approach) %>%
  mutate(Z_resid_lag1 = dplyr::lag(Z_resid)) %>%
  ungroup() %>%
  filter(is.finite(Z_resid_lag1))


# 3) Test velocity on residualized dopamine

velocity_approach_activity_model <- lmer(
  Z_resid ~ Z_resid_lag1 + v_reward_radial * Region * approach * Session +
    (1 | Animal) + (1 | Trial) + (1 | Trial:approach_counter) + (1 | Animal:Site),
  data = df2, REML = FALSE,
  control = lmerControl(calc.derivs = FALSE)
)

slopes <- emtrends(
  velocity_approach_activity_model,
  ~ Region * approach | Session,
  var = "v_reward_radial"
)

velocity_approach_activity_trends <- as.data.frame(test(slopes, adjust = "bonferroni"))
print(velocity_approach_activity_trends)